# تدقيق أثر حملة «ربيع القهوة» على المبيعات: دفتر التحقق وإعادة الإنتاج

يعيد هذا الدفتر حساب **كل رقم** في المذكرة التنفيذية من الملف الخام، ثم يقارنه آلياً بقيمته في المذكرة (القسم الأخير). فإن اختلف رقم واحد توقف التشغيل.

**طريقة التشغيل:** `Kernel ▸ Restart & Run All` من أي مجلد داخل المشروع. يكتشف الدفتر جذر المشروع (المجلد الذي يحوي `data/raw`)، ويقرأ `data/raw/campaign-raw.csv`، ويكتب التقارير في `reports/html/` (ملف لكل قسم + ملخص تنفيذي).

| القسم | المحتوى | يقابل في المذكرة |
|---|---|---|
| 0 | سجل الأرقام والإعدادات وأدوات العرض | ٨ |
| 1 | تحميل البيانات وفحوص السلامة | ٨ |
| 2 | دوال النمو وفرق الفروق (DiD) | ٣ و٤ |
| 3 | إعادة إنتاج حساب التقرير الأصلي (أساس ١٢ شهراً) | ٢ و٤ و٥ |
| 4 | مسار نسبة الإيراد، الاتجاهات المتوازية، والضجيج | ٣ و٤ و٧ |
| 5 | متوسط قيمة الطلب | ٤ و٥ |
| 6 | عدد الطلبات | ٤ و٥ |
| 7 | عتبات الكشف ومدة التجربة ونقطة التعادل | ٦ و٧ |
| 8 | نمو الفروع الفردية | ٤ و٥ |
| 9 | اختبارات تكميلية (اختيارية، معطّلة افتراضياً) | — |
| 10 | تدقيق التطابق مع المذكرة + الملخص التنفيذي | كل الأقسام |

**الأسلوب الإحصائي:** فرق الفروق على متوسطات شهرية لمجموع كل مجموعة، بأساس الأشهر العشرة غير الفعالة:

$$\hat{\tau}=\frac{\bar Y^{T}_{act}-\bar Y^{T}_{base}}{\bar Y^{T}_{base}}-\frac{\bar Y^{C}_{act}-\bar Y^{C}_{base}}{\bar Y^{C}_{base}}$$

**قاعدة العرض العربي:** لا نطبع نصاً عربياً بـ `print`، فالمتصفح وحده يشكّل العربية ويضبط اتجاهها بدقة. كل مخرج عربي يُعرض عبر HTML بخاصية `dir="rtl"`، والرسوم مبنية بـ SVG أصلي.

In [1]:
# =====================================================================
# 0. سجل الأرقام (المصدر الوحيد) + الإعدادات + أدوات العرض
# =====================================================================
import os, html, itertools, math
from pathlib import Path
from statistics import NormalDist
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# False = يطابق نص المذكرة (لم تُجرَ اختبارات دلالة رسمية). True = يضيف القسم 9.
RUN_SENSITIVITY = False

def find_project_root():
    """جذر المشروع = أقرب مجلد (من مجلد التشغيل صعوداً) يحوي data/raw. يعمل من notebooks/ أو من الجذر."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data" / "raw").is_dir():
            return p
    raise FileNotFoundError("project root not found: expected a data/raw folder above the current directory.")

ROOT = find_project_root()
DATA_PATH = Path(os.environ.get("CAMPAIGN_CSV") or ROOT / "data" / "raw" / "campaign-raw.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"missing: {DATA_PATH}")
REPORT_DIR = ROOT / "reports" / "html"          # كل ما يولّده الكود يذهب هنا (يمكن حذفه وإعادة توليده)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

C = dict(month="الشهر", branch="الفرع", in_camp="ضمن فروع الحملة", active="الحملة فعالة هذا الشهر",
         price_cut="خفض سعر عام", orders="عدد الطلبات", aov="متوسط قيمة الطلب", rev="الإيراد")
YES, NO = "نعم", "لا"
EXPECTED = dict(rows=144, cols=8, branches=12, months=12, camp_branches=5, ctrl_branches=7, active_months=2)
OUTLIER_MONTH = "2026-08"   # الشهر الشاذ المذكور في المذكرة (يُتحقق منه آلياً)
K_CONF = 2.0                # تقريب 1.96 (ثقة 95%)
K_POWER80 = 2.8             # تقريب 1.96 + 0.84 (ثقة 95% وقدرة 80%)

# ---- سجل أرقام المذكرة: (القيمة في المذكرة، عدد المنازل العشرية، الوصف، القسم) ----
MEMO = {
 "rows":              (144, 0, "عدد الصفوف", "٨"),
 "max_rounding_gap":  (20.4, 1, "أقصى فرق بين الإيراد وحاصل (الطلبات × متوسط الطلب) بالريال", "٨"),
 "price_cut_branches":(12, 0, "عدد الفروع المشمولة بخفض السعر العام", "٨"),
 "base12_camp":       (1820979, 0, "أساس الـ12 شهراً للخمسة (ريال)", "٥"),
 "base10_camp":       (1699846, 0, "أساس الأشهر العشرة للخمسة (ريال)", "٤"),
 "act_camp":          (2426647, 0, "متوسط شهري الحملة للخمسة (ريال)", "٤"),
 "base10_ctrl":       (1048445, 0, "أساس الأشهر العشرة للسبعة (ريال)", "٤"),
 "act_ctrl":          (1490962, 0, "متوسط شهري الحملة للسبعة (ريال)", "٤"),
 "base_gap_pct":      (7.1, 1, "ارتفاع أساس الـ12 شهراً عن أساس الـ10 (%)", "٥"),
 "growth12_camp":     (33.26, 2, "نمو الخمسة بطريقة التقرير (%)", "٥"),
 "growth12_ctrl":     (32.86, 2, "نمو السبعة بطريقة التقرير (%)", "٥"),
 "report_total_pct":  (33, 0, "رقم التقرير الرئيسي (%)", "١"),
 "g_rev_camp":        (42.76, 2, "نمو إيراد الخمسة بالأساس المصحَّح (%)", "٤"),
 "g_rev_ctrl":        (42.21, 2, "نمو إيراد السبعة بالأساس المصحَّح (%)", "٤"),
 "did_rev":           (0.55, 2, "فرق الفروق للإيراد (نقطة مئوية)", "١"),
 "rel_effect_pct":    (0.4, 1, "التغير النسبي في نسبة الإيراد بين المجموعتين (%)", "١"),
 "aov_base_camp":     (107.92, 2, "متوسط قيمة الطلب للخمسة: الأساس", "٤"),
 "aov_act_camp":      (111.23, 2, "متوسط قيمة الطلب للخمسة: الحملة", "٤"),
 "aov_base_ctrl":     (107.91, 2, "متوسط قيمة الطلب للسبعة: الأساس", "٤"),
 "aov_act_ctrl":      (108.72, 2, "متوسط قيمة الطلب للسبعة: الحملة", "٤"),
 "g_aov_camp":        (3.07, 2, "نمو قيمة الطلب للخمسة (%)", "٤"),
 "g_aov_ctrl":        (0.75, 2, "نمو قيمة الطلب للسبعة (%)", "٤"),
 "did_aov":           (2.32, 2, "فرق الفروق لقيمة الطلب (نقطة مئوية)", "٤"),
 "g_ord_camp":        (38.51, 2, "نمو طلبات الخمسة (%)", "٤"),
 "g_ord_ctrl":        (41.15, 2, "نمو طلبات السبعة (%)", "٤"),
 "did_ord":           (-2.64, 2, "فرق الفروق للطلبات (نقطة مئوية)", "٤"),
 "pre_g_camp":        (-4.61, 2, "تغير إيراد الخمسة من سبتمبر إلى فبراير (%)", "١"),
 "pre_g_ctrl":        (-5.54, 2, "تغير إيراد السبعة من سبتمبر إلى فبراير (%)", "١"),
 "ratio_pre_min":     (1.59, 2, "أدنى نسبة إيراد قبل الحملة", "١"),
 "ratio_pre_max":     (1.63, 2, "أعلى نسبة إيراد قبل الحملة", "١"),
 "ratio_mar":         (1.64, 2, "نسبة الإيراد: مارس", "٥"),
 "ratio_apr":         (1.62, 2, "نسبة الإيراد: أبريل", "٥"),
 "ratio_may":         (1.65, 2, "نسبة الإيراد: مايو", "٥"),
 "ratio_jun":         (1.59, 2, "نسبة الإيراد: يونيو", "٥"),
 "ratio_jul":         (1.60, 2, "نسبة الإيراد: يوليو", "٥"),
 "ratio_aug":         (1.71, 2, "نسبة الإيراد: أغسطس", "٥"),
 "cv_base":           (2.2, 1, "الانحراف المعياري لنسبة الإيراد في الأشهر العشرة (%)", "٨"),
 "cv_no_aug":         (1.3, 1, "الانحراف المعياري دون أغسطس (%)", "٨"),
 "se_memo":           (1.55, 2, "الخطأ المعياري لمتوسط شهرين (%)", "٣"),
 "se_no_aug":         (0.9, 1, "الخطأ المعياري دون أغسطس (%)", "٨"),
 "z_rev":             (0.24, 2, "بُعد شهري الحملة عن الأساس بالخطأ المعياري (نسبة الإيراد)", "١"),
 "thr_conf":          (3, 0, "أصغر أثر نسبي ممكن التمييز بشهرين (%)", "١"),
 "thr_power80":       (4, 0, "أصغر أثر نسبي بقدرة 80% بشهرين (%)", "٧"),
 "thr_no_aug":        (2, 0, "العتبة بعد استبعاد أغسطس (%)", "٨"),
 "aov_base_mean":     (-0.18, 2, "فرق AOV بين المجموعتين: متوسط الأساس (ريال)", "٤"),
 "aov_base_sd":       (4.03, 2, "فرق AOV: الانحراف المعياري (ريال)", "٤"),
 "aov_act_mean":      (2.51, 2, "فرق AOV: متوسط شهري الحملة (ريال)", "٤"),
 "aov_dist":          (2.69, 2, "فرق AOV: المسافة عن الأساس (ريال)", "٤"),
 "aov_se":            (2.85, 2, "فرق AOV: الخطأ المعياري (ريال)", "٤"),
 "aov_z":             (0.95, 2, "فرق AOV: المسافة بالخطأ المعياري (٠٫٩٤٥٣ بلا تقريب وسيط؛ كانت المذكرة ٠٫٩٤ من قسمة ٢٫٦٩ ÷ ٢٫٨٥ المقرَّبتين)", "٤ و٥"),
 "ord_base_mean":     (1.6239, 4, "نسبة الطلبات: متوسط الأساس", "٤"),
 "ord_base_sd":       (0.0668, 4, "نسبة الطلبات: الانحراف المعياري", "٤"),
 "ord_se":            (0.0472, 4, "نسبة الطلبات: الخطأ المعياري", "٤"),
 "ord_act_mean":      (1.5935, 4, "نسبة الطلبات: متوسط شهري الحملة", "٤"),
 "ord_dist":          (-0.0305, 4, "نسبة الطلبات: المسافة عن الأساس", "٤"),
 "ord_z":             (-0.64, 2, "نسبة الطلبات: المسافة بالخطأ المعياري", "٥"),
 "camp_min":          (38.20, 2, "أدنى نمو بين الفروع الخمسة (%)", "٤"),
 "camp_max":          (50.16, 2, "أعلى نمو بين الفروع الخمسة (%)", "٤"),
 "ctrl_min":          (37.68, 2, "أدنى نمو بين الفروع السبعة (%)", "٤"),
 "ctrl_max":          (49.61, 2, "أعلى نمو بين الفروع السبعة (%)", "٤"),
 "total_ratio_1d":    (1.6, 1, "نسبة مبيعات الخمسة إلى السبعة في التقرير", "٥"),
 "total_ratio":       (1.62, 2, "نسبة مبيعات الخمسة إلى السبعة (إجمالي السنة)", "٧"),
 "per_branch_ratio":  (2.3, 1, "نسبة متوسط الفرع (الخمسة ÷ السبعة)", "٧"),
 "n_conf_3": (3, 0, "أشهر لكشف أثر 3% (ثقة 95%، قدرة ≈50%)", "٦"),
 "n_conf_2": (5, 0, "أشهر لكشف أثر 2% (ثقة 95%، قدرة ≈50%)", "٦"),
 "n_conf_15": (9, 0, "أشهر لكشف أثر 1.5% (ثقة 95%، قدرة ≈50%)", "٦"),
 "n_pow_3": (5, 0, "أشهر لكشف أثر 3% (قدرة 80%)", "٦"),
 "n_pow_2": (10, 0, "أشهر لكشف أثر 2% (قدرة 80%)", "٦"),
 "n_pow_15": (17, 0, "أشهر لكشف أثر 1.5% (قدرة 80%)", "٦"),
}
# جدول التقرير الأصلي (القسم 3): الفرع: (متوسط 12 شهراً، متوسط شهري الحملة، نسبة التغير %)
REPORT_TABLE = {"الرياض ١": (460706, 627373, 36), "الرياض ٢": (401333, 523200, 30),
                "جدة ١": (357893, 465008, 30), "جدة ٢": (320136, 421785, 32), "الدمام ١": (280911, 389281, 39)}

CHECKS = []
def add_check(label, section, memo_val, value, dec):
    got = round(float(value), dec)
    CHECKS.append(dict(label=label, section=section, memo=memo_val, got=got, ok=abs(got - memo_val) < 1e-9))
def check(key, value):
    memo_val, dec, label, sec = MEMO[key]
    add_check(label, sec, memo_val, value, dec)

# ---------------------------- أدوات HTML (RTL) ----------------------------
_CSS = """<style>
.ca{font-family:"Segoe UI","Noto Naskh Arabic","Noto Sans Arabic",Tahoma,Arial,sans-serif;direction:rtl;text-align:right;
 background:#f6f7fb;color:#1c2333;padding:18px 20px;border-radius:10px;line-height:1.75}
.ca h1{font-size:21px;margin:0 0 4px}.ca h2{font-size:16px;margin:22px 0 8px;color:#33415c}
.ca .sub{color:#667;font-size:13px;margin-bottom:14px}
.ca .cards{display:flex;gap:12px;flex-wrap:wrap}
.ca .card{background:#fff;border-radius:10px;padding:12px 16px;min-width:170px;box-shadow:0 1px 3px #0001;flex:1}
.ca .card .k{font-size:12px;color:#667}.ca .card .v{font-size:21px;font-weight:600;margin-top:2px}
.ca .good{color:#1b7f4b}.ca .bad{color:#b3261e}.ca .warn{color:#9a6700}
.ca table{border-collapse:collapse;background:#fff;font-size:13px;box-shadow:0 1px 3px #0001}
.ca th,.ca td{padding:6px 12px;border-bottom:1px solid #eef0f5;text-align:center;white-space:nowrap}
.ca th{background:#33415c;color:#fff;font-weight:600}
.ca td.lbl{text-align:right;font-weight:600}
.ca td.num,.ca bdi{direction:ltr;unicode-bidi:isolate;font-variant-numeric:tabular-nums}
.ca .note{background:#fff8e1;border-right:4px solid #f0b429;padding:9px 13px;border-radius:6px;font-size:13px;margin:12px 0}
.ca .ok{color:#1b7f4b;font-weight:700}.ca .no{color:#b3261e;font-weight:700}
.ca .scroll{overflow-x:auto}
.ca .chart{position:relative;background:#fff;border-radius:8px;box-shadow:0 1px 3px #0001;margin:8px 0;direction:ltr}
.ca .cl{position:absolute;white-space:nowrap;line-height:1.2;unicode-bidi:plaintext;font-family:inherit}
</style>"""

def num(x, d=2, sign=False):
    return f'<bdi dir="ltr">{x:+,.{d}f}</bdi>' if sign else f'<bdi dir="ltr">{x:,.{d}f}</bdi>'

def kpi(items):
    """items: [(العنوان، قيمة HTML، فئة CSS)]"""
    return '<div class="cards">' + "".join(
        f'<div class="card"><div class="k">{html.escape(k)}</div><div class="v {c}">{v}</div></div>'
        for k, v, c in items) + "</div>"

def note(text): return f'<div class="note">{text}</div>'
def h2(text): return f"<h2>{html.escape(text)}</h2>"

def table_html(frame, d=2):
    """جدول RTL: الخلايا الرقمية LTR معزولة (لا ينقلب الإشارة +/-)، d = رقم أو قاموس {عمود: منازل}."""
    cols = list(frame.columns)
    head = "".join(f"<th>{html.escape(str(c))}</th>" for c in [frame.index.name or ""] + cols)
    rows = []
    for idx, r in frame.iterrows():
        cells = [f'<td class="lbl">{html.escape(str(idx))}</td>']
        for c in cols:
            v = r[c]
            if isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, (bool, np.bool_)):
                dd = d.get(c, 2) if isinstance(d, dict) else d
                cells.append(f'<td class="num">{v:,.{dd}f}</td>')
            else:
                cells.append(f"<td>{html.escape(str(v))}</td>")
        rows.append("<tr>" + "".join(cells) + "</tr>")
    return f'<div class="scroll"><table><thead><tr>{head}</tr></thead><tbody>{"".join(rows)}</tbody></table></div>'

# ---------------------------- رسوم: أشكال SVG + نصوص HTML ----------------------------
# الأشكال (خطوط/أعمدة) SVG، أما النصوص العربية فعناصر HTML مُموضعة فوقها: يشكّلها المتصفح ويضبط اتجاهها بدقة.
class _Chart:
    def __init__(self, w, h):
        self.w, self.h, self.shapes, self.labels = w, h, [], []

    def shape(self, svg):
        self.shapes.append(svg)

    def text(self, x, y, s, anchor="middle", size=11, fill="#667", weight="400", rot=None):
        tx = {"middle": "-50%", "end": "-100%", "start": "0"}[anchor]
        rt = f" rotate({rot}deg)" if rot else ""
        self.labels.append(
            f'<div class="cl" style="left:{x / self.w * 100:.2f}%;top:{y / self.h * 100:.2f}%;'
            f'transform:translate({tx},-70%){rt};transform-origin:{"100%" if anchor == "end" else "50%"} 70%;'
            f'font-size:{size}px;color:{fill};font-weight:{weight}">{html.escape(str(s))}</div>')

    def render(self):
        svg = (f'<svg viewBox="0 0 {self.w} {self.h}" xmlns="http://www.w3.org/2000/svg" role="img" '
               f'style="display:block;width:100%;height:auto">{"".join(self.shapes)}</svg>')
        return f'<div class="chart" style="max-width:{self.w}px">{svg}{"".join(self.labels)}</div>'

def _range(vals, zero=False, pad=0.1):
    lo, hi = min(vals), max(vals)
    if zero: lo, hi = min(lo, 0), max(hi, 0)
    span = (hi - lo) or 1.0
    return lo - span * pad, hi + span * pad

def _legend(ch, names, colors):
    x = ch.w - 14
    for name, col in zip(names, colors):
        ch.text(x - 16, 48, name, anchor="end", size=11, fill="#33415c")
        ch.shape(f'<rect x="{x-12:.1f}" y="43" width="10" height="10" rx="2" fill="{col}"/>')
        x -= 30 + 7 * len(name)

def svg_bars(cats, series, colors, title, bar_colors=None, band=None, d=1, rot=0, w=760, h=300):
    ml, mr, mt, mb = 56, 14, (62 if len(series) > 1 else 44), (66 if rot else 46)
    vals = [v for s in series.values() for v in s] + (list(band) if band else [])
    lo, hi = _range(vals, zero=True)
    Y = lambda v: mt + (hi - v) / (hi - lo) * (h - mt - mb)
    n, slot = len(series), (w - ml - mr) / len(cats)
    bw = slot * 0.7 / n
    ch = _Chart(w, h)
    ch.text(w / 2, 22, title, size=14, fill="#33415c", weight="600")
    if band:
        ch.shape(f'<rect x="{ml}" y="{Y(band[1]):.1f}" width="{w-ml-mr}" height="{Y(band[0])-Y(band[1]):.1f}" fill="#9aa5b8" opacity=".18"/>')
    for i in range(5):
        v = lo + (hi - lo) * i / 4
        ch.shape(f'<line x1="{ml}" x2="{w-mr}" y1="{Y(v):.1f}" y2="{Y(v):.1f}" stroke="#eef0f5"/>')
        ch.text(ml - 6, Y(v), f"{v:,.{d}f}", anchor="end")
    ch.shape(f'<line x1="{ml}" x2="{w-mr}" y1="{Y(0):.1f}" y2="{Y(0):.1f}" stroke="#667"/>')
    for gi, (name, vs) in enumerate(series.items()):
        for ci, v in enumerate(vs):
            x = ml + ci * slot + slot * 0.15 + gi * bw
            col = bar_colors[ci] if bar_colors else colors[gi]
            y0, y1 = min(Y(v), Y(0)), max(Y(v), Y(0))
            ch.shape(f'<rect x="{x:.1f}" y="{y0:.1f}" width="{bw-2:.1f}" height="{max(y1-y0,0.5):.1f}" rx="2" fill="{col}"/>')
            ch.text(x + bw / 2 - 1, (y0 - 8) if v >= 0 else (y1 + 9), f"{v:,.{d}f}", size=10, fill="#33415c")
    for ci, c in enumerate(cats):
        cx = ml + ci * slot + slot / 2
        ch.text(cx + (6 if rot else 0), h - mb + 18, c, anchor=("end" if rot else "middle"), rot=(-35 if rot else None))
    if n > 1: _legend(ch, list(series), colors)
    return ch.render()

def svg_lines(x, series, colors, title, band=None, shade=None, d=3, w=760, h=300):
    ml, mr, mt, mb = 60, 14, (62 if len(series) > 1 else 44), 62
    vals = [v for s in series.values() for v in s] + (list(band) if band else [])
    lo, hi = _range(vals, pad=0.12)
    X = lambda i: ml + (i + 0.5) * (w - ml - mr) / len(x)
    Y = lambda v: mt + (hi - v) / (hi - lo) * (h - mt - mb)
    ch = _Chart(w, h)
    ch.text(w / 2, 22, title, size=14, fill="#33415c", weight="600")
    if band:
        ch.shape(f'<rect x="{ml}" y="{Y(band[1]):.1f}" width="{w-ml-mr}" height="{Y(band[0])-Y(band[1]):.1f}" fill="#9aa5b8" opacity=".18"/>')
    for i in shade or []:
        sw = (w - ml - mr) / len(x)
        ch.shape(f'<rect x="{X(i)-sw/2:.1f}" y="{mt}" width="{sw:.1f}" height="{h-mt-mb}" fill="#d1495b" opacity=".10"/>')
    for i in range(5):
        v = lo + (hi - lo) * i / 4
        ch.shape(f'<line x1="{ml}" x2="{w-mr}" y1="{Y(v):.1f}" y2="{Y(v):.1f}" stroke="#eef0f5"/>')
        ch.text(ml - 6, Y(v), f"{v:,.{d}f}", anchor="end")
    for gi, (name, vs) in enumerate(series.items()):
        pts = " ".join(f"{X(i):.1f},{Y(v):.1f}" for i, v in enumerate(vs))
        ch.shape(f'<polyline points="{pts}" fill="none" stroke="{colors[gi]}" stroke-width="2.2"/>')
        for i, v in enumerate(vs):
            ch.shape(f'<circle cx="{X(i):.1f}" cy="{Y(v):.1f}" r="3" fill="{colors[gi]}"/>')
    for i, lab in enumerate(x):
        ch.text(X(i) + 6, h - mb + 18, lab, anchor="end", rot=-35)
    if len(series) > 1: _legend(ch, list(series), colors)
    return ch.render()

def svg_hbars(labels, values, colors, title, d=2, w=760):
    row, mt, lab, ml = 26, 44, 120, 16
    h = mt + row * len(labels) + 14
    vmax = max(values) * 1.12
    X = lambda v: ml + v / vmax * (w - ml - lab - 70)
    ch = _Chart(w, h)
    ch.text(w / 2, 22, title, size=14, fill="#33415c", weight="600")
    for i, (l, v, c) in enumerate(zip(labels, values, colors)):
        y = mt + i * row
        ch.shape(f'<rect x="{ml}" y="{y}" width="{X(v)-ml:.1f}" height="{row-6}" rx="3" fill="{c}"/>')
        ch.text(X(v) + 6, y + (row - 6) / 2, f"{v:,.{d}f}", anchor="start", size=11, fill="#33415c")
        ch.text(w - 8, y + (row - 6) / 2, l, anchor="end", size=12, fill="#1c2333")
    return ch.render()

def emit(section, title, blocks, subtitle=""):
    """يعرض التقرير داخل الدفتر ويحفظه ملف HTML مستقلاً في reports/."""
    inner = (f'<div class="ca"><h1>{html.escape(title)}</h1><div class="sub">{html.escape(subtitle)}</div>'
             f'{"".join(blocks)}</div>')
    display(HTML(_CSS + inner))
    page = (f'<!doctype html><html lang="ar" dir="rtl"><head><meta charset="utf-8">'
            f'<meta name="viewport" content="width=device-width, initial-scale=1">'
            f'<title>{html.escape(title)}</title>{_CSS}</head><body style="margin:0;background:#f6f7fb">{inner}</body></html>')
    path = REPORT_DIR / f"analysis_report_{section}.html"
    path.write_text(page, encoding="utf-8")
    print("saved:", path.relative_to(ROOT))

print("setup ok | root:", ROOT.name, "| data:", DATA_PATH.relative_to(ROOT) if DATA_PATH.is_relative_to(ROOT) else DATA_PATH, "| sensitivity tests:", RUN_SENSITIVITY)

setup ok | root: coffee-spring-campaign-review | data: data\raw\campaign-raw.csv | sensitivity tests: False


## 1. تحميل البيانات وفحوص السلامة
نتحقق (لا نفترض): الأبعاد، القيم المفقودة، اكتمال ١٢×١٢، عدم التكرار، شهران فعّالان فقط، ٥ فروع مشمولة، اتساق `الإيراد ≈ الطلبات × متوسط الطلب`، وأن خفض السعر العام شمل الفروع الاثني عشر في شهري الحملة وحدهما.

In [2]:
# =====================================================================
# 1. تحميل البيانات + فحوص السلامة + تعريف الفترات والمجموعات (مرة واحدة)
# =====================================================================
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

assert df.shape == (EXPECTED["rows"], EXPECTED["cols"]), f"shape={df.shape}"
assert df.isnull().sum().sum() == 0, "missing values"
assert df[C["month"]].nunique() == EXPECTED["months"]
assert df[C["branch"]].nunique() == EXPECTED["branches"]
assert (df.groupby(C["branch"])[C["month"]].nunique() == EXPECTED["months"]).all(), "branch without 12 months"
assert not df.duplicated([C["branch"], C["month"]]).any(), "duplicate (branch, month)"

MONTHS   = sorted(df[C["month"]].unique())                      # صيغة YYYY-MM: الترتيب النصي = الزمني
ACTIVE   = sorted(df.loc[df[C["active"]] == YES, C["month"]].unique())
BASELINE = [m for m in MONTHS if m not in ACTIVE]
PRE      = [m for m in BASELINE if m < ACTIVE[0]]              # قبل الحملة (اختبار الاتجاهات المتوازية)
POST     = [m for m in BASELINE if m > ACTIVE[-1]]             # بعد الحملة
assert len(ACTIVE) == EXPECTED["active_months"], ACTIVE

CAMP = df[df[C["in_camp"]] == YES].copy()
CTRL = df[df[C["in_camp"]] == NO].copy()
assert CAMP[C["branch"]].nunique() == EXPECTED["camp_branches"]
assert CTRL[C["branch"]].nunique() == EXPECTED["ctrl_branches"]

calc = df[C["orders"]] * df[C["aov"]]
gap = (calc - df[C["rev"]]).abs()
rel_err = gap / df[C["rev"]]
assert rel_err.max() < 0.005, "revenue != orders x AOV beyond rounding tolerance"

pc = df[df[C["price_cut"]] == YES]
price_cut_uniform = (sorted(pc[C["month"]].unique()) == ACTIVE) and pc[C["branch"]].nunique() == EXPECTED["branches"]
assert price_cut_uniform, "price cut is not uniform across all branches in the campaign months"

check("rows", len(df)); check("max_rounding_gap", gap.max()); check("price_cut_branches", pc[C["branch"]].nunique())

emit("1_integrity", "فحص السلامة الهيكلية", [
    kpi([("الصفوف × الأعمدة", num(len(df), 0) + " × " + num(df.shape[1], 0), ""),
         ("الفروع (حملة / ضابطة)", f'{num(CAMP[C["branch"]].nunique(),0)} / {num(CTRL[C["branch"]].nunique(),0)}', ""),
         ("الأشهر", num(len(MONTHS), 0), ""),
         ("أقصى فرق مطلق (ريال)", num(gap.max(), 1), "good"),
         ("أقصى خطأ نسبي", num(rel_err.max() * 100, 3) + "%", "good")]),
    h2("التعريفات المستخرجة آلياً من البيانات"),
    table_html(pd.DataFrame({"القيمة": [", ".join(ACTIVE), f"{len(BASELINE)} أشهر", f"{len(PRE)} أشهر ({PRE[0]} إلى {PRE[-1]})",
                                       f"{len(POST)} أشهر ({POST[0]} إلى {POST[-1]})", "نعم" if price_cut_uniform else "لا"]},
               index=pd.Index(["أشهر الحملة", "خط الأساس", "ما قبل الحملة", "ما بعد الحملة", "خفض السعر شامل للفروع الاثني عشر في شهري الحملة فقط؟"], name="البند"))),
    note("خفض السعر العام شمل الفروع كلها في شهري الحملة، فتمتصّه المجموعة الضابطة في فرق الفروق بشرط توازي المسارين قبل الحملة. "
         "لكنه يجعل التقدير خاصاً بـ «الحملة مع خفض السعر»، ولا يعزل أثر الحملة وحدها."),
], subtitle=f"أشهر الحملة: {', '.join(ACTIVE)}")

البند,القيمة
أشهر الحملة,"2026-03, 2026-04"
خط الأساس,10 أشهر
ما قبل الحملة,6 أشهر (2025-09 إلى 2026-02)
ما بعد الحملة,4 أشهر (2026-05 إلى 2026-08)
خفض السعر شامل للفروع الاثني عشر في شهري الحملة فقط؟,نعم


saved: reports\html\analysis_report_1_integrity.html


## 2. دوال النمو وفرق الفروق (مكان واحد لكل المعادلات)
**متوسط قيمة الطلب** لمجموعة وفترة = ΣالإيرادΣ ÷ ΣالطلباتΣ (متوسط موزون، لا متوسط متوسطات). وبهذا يصبح تفكيك الإيراد متطابقة جبرية:

$$1+g_{rev}=(1+g_{orders})(1+g_{aov})$$

ولذلك **لا تُجمَع فروق النقاط المئوية** (‎-2.64 + 2.32 ≠ 0.55)؛ التجميع الدقيق يكون على اللوغاريتم.

In [3]:
# =====================================================================
# 2. دوال الحساب الأساسية + فرق الفروق (DiD)
# =====================================================================
def monthly_mean(frame, col, months):
    """نجمع الفروع داخل كل شهر، ثم نأخذ متوسط الأشهر."""
    sub = frame[frame[C["month"]].isin(months)]
    return sub.groupby(C["month"])[col].sum().mean()

def weighted_aov(frame, months):
    sub = frame[frame[C["month"]].isin(months)]
    return sub[C["rev"]].sum() / sub[C["orders"]].sum()

def growth(a, b):
    return a / b - 1

def group_metrics(frame, base=BASELINE, act=ACTIVE):
    m = {}
    for col, key in ((C["rev"], "rev"), (C["orders"], "orders")):
        m[f"{key}_base"] = monthly_mean(frame, col, base)
        m[f"{key}_act"]  = monthly_mean(frame, col, act)
        m[f"g_{key}"]    = growth(m[f"{key}_act"], m[f"{key}_base"])
    m["aov_base"], m["aov_act"] = weighted_aov(frame, base), weighted_aov(frame, act)
    m["g_aov"] = growth(m["aov_act"], m["aov_base"])
    return m

M_CAMP, M_CTRL = group_metrics(CAMP), group_metrics(CTRL)

for name, M in (("camp", M_CAMP), ("ctrl", M_CTRL)):      # التفكيك متطابقة جبرية
    assert np.isclose((1 + M["g_orders"]) * (1 + M["g_aov"]) - 1, M["g_rev"], atol=1e-10), name

LABELS = {"rev": "الإيراد", "orders": "عدد الطلبات", "aov": "متوسط قيمة الطلب"}
rows = []
for key, label in LABELS.items():
    gt, gc = M_CAMP[f"g_{key}"] * 100, M_CTRL[f"g_{key}"] * 100             # gt: الحملة، gc: الضابطة
    rel = ((1 + M_CAMP[f"g_{key}"]) / (1 + M_CTRL[f"g_{key}"]) - 1) * 100
    lg  = math.log((1 + M_CAMP[f"g_{key}"]) / (1 + M_CTRL[f"g_{key}"])) * 100
    rows.append([label, gt, gc, gt - gc, rel, lg])
DID = pd.DataFrame(rows, columns=["المقياس", "نمو الحملة %", "نمو الضابطة %", "فرق الفروق (نقطة مئوية)",
                                  "الأثر النسبي %", "الأثر اللوغاريتمي ×100"]).set_index("المقياس")

log_o, log_a, log_r = (DID.loc[LABELS[k], "الأثر اللوغاريتمي ×100"] for k in ("orders", "aov", "rev"))
assert np.isclose(log_o + log_a, log_r, atol=1e-8)
pp_sum = DID.loc[LABELS["orders"], "فرق الفروق (نقطة مئوية)"] + DID.loc[LABELS["aov"], "فرق الفروق (نقطة مئوية)"]

R = LABELS["rev"]; O = LABELS["orders"]; A = LABELS["aov"]
check("base10_camp", M_CAMP["rev_base"]); check("act_camp", M_CAMP["rev_act"])
check("base10_ctrl", M_CTRL["rev_base"]); check("act_ctrl", M_CTRL["rev_act"])
check("g_rev_camp", DID.loc[R, "نمو الحملة %"]); check("g_rev_ctrl", DID.loc[R, "نمو الضابطة %"])
check("did_rev", DID.loc[R, "فرق الفروق (نقطة مئوية)"])
check("aov_base_camp", M_CAMP["aov_base"]); check("aov_act_camp", M_CAMP["aov_act"])
check("aov_base_ctrl", M_CTRL["aov_base"]); check("aov_act_ctrl", M_CTRL["aov_act"])
check("g_aov_camp", DID.loc[A, "نمو الحملة %"]); check("g_aov_ctrl", DID.loc[A, "نمو الضابطة %"])
check("did_aov", DID.loc[A, "فرق الفروق (نقطة مئوية)"])
check("g_ord_camp", DID.loc[O, "نمو الحملة %"]); check("g_ord_ctrl", DID.loc[O, "نمو الضابطة %"])
check("did_ord", DID.loc[O, "فرق الفروق (نقطة مئوية)"])

emit("2_did", "فرق الفروق (DiD) وتفكيك الإيراد", [
    kpi([("فرق الفروق: الإيراد", num(DID.loc[R, "فرق الفروق (نقطة مئوية)"], 2, True) + " نقطة", "warn"),
         ("فرق الفروق: الطلبات", num(DID.loc[O, "فرق الفروق (نقطة مئوية)"], 2, True) + " نقطة", "bad"),
         ("فرق الفروق: قيمة الطلب", num(DID.loc[A, "فرق الفروق (نقطة مئوية)"], 2, True) + " نقطة", "good"),
         ("الأثر النسبي للإيراد", num(DID.loc[R, "الأثر النسبي %"], 2, True) + "%", "warn")]),
    h2("جدول النتائج"), table_html(DID, 3),
    h2("الرسوم"),
    svg_bars(list(LABELS.values()), {"الخمسة (الحملة)": DID["نمو الحملة %"].tolist(), "السبعة (الضابطة)": DID["نمو الضابطة %"].tolist()},
             ["#2a6f97", "#a9b4c2"], "نمو كل مجموعة مقابل خط الأساس"),
    svg_bars(["عدد الطلبات", "متوسط قيمة الطلب", "الإيراد"], {"التفكيك": [log_o, log_a, log_r]}, ["#33415c"],
             "التفكيك اللوغاريتمي لفرق الفروق", bar_colors=["#d1495b", "#3a7d44", "#33415c"], d=3),
    note(f"مجموع فرقي النقاط المئوية للطلبات وقيمة الطلب = {num(pp_sum,2,True)} ويختلف عن فرق الإيراد {num(DID.loc[R,'فرق الفروق (نقطة مئوية)'],2,True)}؛ "
         f"ليس خطأ حسابياً، فالنقاط المئوية لا تتجمع في علاقة ضرب. على اللوغاريتم: {num(log_o,3,True)} + {num(log_a,3,True)} = {num(log_r,3,True)}."),
], subtitle=f"خط الأساس: {len(BASELINE)} أشهر | الحملة: {' و '.join(ACTIVE)} | الأزرق: الخمسة، الرمادي: السبعة")

المقياس,نمو الحملة %,نمو الضابطة %,فرق الفروق (نقطة مئوية),الأثر النسبي %,الأثر اللوغاريتمي ×100
الإيراد,42.757,42.207,0.550,0.387,0.386
عدد الطلبات,38.510,41.153,-2.643,-1.872,-1.890
متوسط قيمة الطلب,3.066,0.747,2.319,2.302,2.276


saved: reports\html\analysis_report_2_did.html


## 3. إعادة إنتاج حساب التقرير الأصلي (أساس ١٢ شهراً)
نطبّق **طريقة التقرير نفسها** (متوسط الأشهر الاثني عشر أساساً) على المجموعتين. هذا هو الدليل الأقوى: السبعة التي لم تتلقَّ الحملة تعطي رقماً قريباً من رقم الخمسة. ونتحقق كذلك من أن حسابات التقرير الحسابية سليمة (جدول التقرير فرعاً فرعاً).

In [4]:
# =====================================================================
# 3. طريقة التقرير نفسها على الخمسة والسبعة + مطابقة جدول التقرير
# =====================================================================
M12_CAMP = group_metrics(CAMP, base=MONTHS)
M12_CTRL = group_metrics(CTRL, base=MONTHS)
base_gap_pct = (M12_CAMP["rev_base"] / M_CAMP["rev_base"] - 1) * 100

check("base12_camp", M12_CAMP["rev_base"]); check("base_gap_pct", base_gap_pct)
check("growth12_camp", M12_CAMP["g_rev"] * 100); check("growth12_ctrl", M12_CTRL["g_rev"] * 100)
check("report_total_pct", M12_CAMP["g_rev"] * 100)
check("aov_base_camp", M_CAMP["aov_base"])  # رقم التقرير 107.92 يطابق أساس الأشهر العشرة

b12 = (CAMP.groupby(C["branch"])[C["rev"]].mean().rename("avg12").to_frame()
       .join(CAMP[CAMP[C["month"]].isin(ACTIVE)].groupby(C["branch"])[C["rev"]].mean().rename("act")))
b12["pct"] = (b12["act"] / b12["avg12"] - 1) * 100
rep_rows = []
for br, (a12, act, pct) in REPORT_TABLE.items():
    add_check(f"جدول التقرير: {br} (متوسط 12 شهراً)", "٥", a12, b12.loc[br, "avg12"], 0)
    add_check(f"جدول التقرير: {br} (متوسط شهري الحملة)", "٥", act, b12.loc[br, "act"], 0)
    add_check(f"جدول التقرير: {br} (نسبة التغير %)", "٥", pct, b12.loc[br, "pct"], 0)
    rep_rows.append([br, b12.loc[br, "avg12"], a12, b12.loc[br, "act"], act, b12.loc[br, "pct"], pct])
REP = pd.DataFrame(rep_rows, columns=["الفرع", "متوسط 12 شهراً (محسوب)", "(التقرير)", "متوسط شهري الحملة (محسوب)", "(التقرير)",
                                      "التغير % (محسوب)", "(التقرير %)"]).set_index("الفرع")

cmp12 = pd.DataFrame({
    "أساس 12 شهراً": [M12_CAMP["rev_base"], M12_CTRL["rev_base"]],
    "أساس 10 أشهر": [M_CAMP["rev_base"], M_CTRL["rev_base"]],
    "متوسط شهري الحملة": [M_CAMP["rev_act"], M_CTRL["rev_act"]],
    "النمو بأساس 12 (طريقة التقرير) %": [M12_CAMP["g_rev"] * 100, M12_CTRL["g_rev"] * 100],
    "النمو بأساس 10 (مصحَّح) %": [M_CAMP["g_rev"] * 100, M_CTRL["g_rev"] * 100],
}, index=pd.Index(["الخمسة (الحملة)", "السبعة (الضابطة)"], name="المجموعة"))

emit("3_report_method", "إعادة إنتاج حساب التقرير الأصلي", [
    kpi([("الخمسة بطريقة التقرير", num(M12_CAMP["g_rev"] * 100, 2) + "%", "warn"),
         ("السبعة بطريقة التقرير (بلا حملة)", num(M12_CTRL["g_rev"] * 100, 2) + "%", "warn"),
         ("أساس 12 شهراً أعلى من أساس 10 بنحو", num(base_gap_pct, 1) + "%", ""),
         ("الخمسة بالأساس المصحَّح", num(M_CAMP["g_rev"] * 100, 2) + "%", "")]),
    h2("الطريقتان على المجموعتين"), table_html(cmp12, {"أساس 12 شهراً": 0, "أساس 10 أشهر": 0, "متوسط شهري الحملة": 0}),
    h2("مطابقة جدول التقرير فرعاً فرعاً"), table_html(REP, {"متوسط 12 شهراً (محسوب)": 0, "(التقرير)": 0, "متوسط شهري الحملة (محسوب)": 0,
                                                       "التغير % (محسوب)": 1, "(التقرير %)": 0}),
    note("حسابات التقرير سليمة. العيب في تصميم المقارنة: خط الأساس يشمل شهري الحملة (فأنقص الرقم)، ولا مجموعة ضابطة. "
         "والفروع السبعة بلا حملة تعطي رقماً قريباً جداً من رقم الخمسة بطريقة التقرير نفسها."),
])

saved: reports\html\analysis_report_3_report_method.html


## 4. مسار نسبة الإيراد، الاتجاهات المتوازية، والضجيج
نسبة الإيراد = إيراد الخمسة ÷ إيراد السبعة في كل شهر: ما يؤثر في المجموعتين معاً (الموسم والخفض العام) يُلغى في النسبة.

**تعريفان للخطأ المعياري، نعرضهما معاً:**
- **اصطلاح المذكرة:** SE = s ÷ √n، ويعامل متوسط الأساس قيمةً معلومة.
- **التعريف المتحفظ:** SE = s × √(1/n_act + 1/n_base)، ويأخذ عدم يقين متوسط الأساس (عشرة أشهر فقط) بالحسبان. وهو أكبر بنحو ١٢٪، ولا يغيّر الاستنتاج.

In [5]:
# =====================================================================
# 4. مسار نسبة الإيراد + الاتجاهات المتوازية + الضجيج
# =====================================================================
def distance_test(series, base_months, act_months):
    """مسافة متوسط شهري الحملة عن متوسط الأساس، مقيسة بخطأين معياريين (اصطلاح المذكرة والمتحفظ)."""
    b, a = series.loc[base_months], series.loc[act_months]
    mu_b, sd_b, mu_a = b.mean(), b.std(ddof=1), a.mean()
    dist = mu_a - mu_b
    se_memo = sd_b / math.sqrt(len(a))
    se_cons = sd_b * math.sqrt(1 / len(a) + 1 / len(b))
    return dict(mu_b=mu_b, sd_b=sd_b, mu_a=mu_a, dist=dist, se_memo=se_memo, z_memo=dist / se_memo,
                se_cons=se_cons, z_cons=dist / se_cons, n_b=len(b), n_a=len(a))

rev_t = CAMP.groupby(C["month"])[C["rev"]].sum()
rev_c = CTRL.groupby(C["month"])[C["rev"]].sum()
ratio = (rev_t / rev_c).rename("نسبة الإيراد (5÷7)")

RT = distance_test(ratio, BASELINE, ACTIVE)
cv_pct = RT["sd_b"] / RT["mu_b"] * 100
se_pct = RT["se_memo"] / RT["mu_b"] * 100                 # الخطأ المعياري لمتوسط شهرين (%)
se_pct_cons = RT["se_cons"] / RT["mu_b"] * 100
rel_effect_pct = (RT["mu_a"] / RT["mu_b"] - 1) * 100       # التغير النسبي في النسبة

# الشهر الشاذ: نتحقق أنه الأبعد عن المتوسط فعلاً قبل استبعاده في تحليل الحساسية
zs = ((ratio.loc[BASELINE] - RT["mu_b"]) / RT["sd_b"]).abs()
assert zs.idxmax() == OUTLIER_MONTH, f"most extreme baseline month is {zs.idxmax()}"
BASE_NO_OUT = [m for m in BASELINE if m != OUTLIER_MONTH]
RT_NA = distance_test(ratio, BASE_NO_OUT, ACTIVE)
cv_na = RT_NA["sd_b"] / RT_NA["mu_b"] * 100
se_na = RT_NA["se_memo"] / RT_NA["mu_b"] * 100

pre = ratio.loc[PRE]
g_pre_t = (rev_t.loc[PRE[-1]] / rev_t.loc[PRE[0]] - 1) * 100
g_pre_c = (rev_c.loc[PRE[-1]] / rev_c.loc[PRE[0]] - 1) * 100
total_ratio = CAMP[C["rev"]].sum() / CTRL[C["rev"]].sum()

check("pre_g_camp", g_pre_t); check("pre_g_ctrl", g_pre_c)
check("ratio_pre_min", pre.min()); check("ratio_pre_max", pre.max())
for k, m in (("ratio_mar", "2026-03"), ("ratio_apr", "2026-04"), ("ratio_may", "2026-05"),
             ("ratio_jun", "2026-06"), ("ratio_jul", "2026-07"), ("ratio_aug", "2026-08")):
    check(k, ratio.loc[m])
check("cv_base", cv_pct); check("cv_no_aug", cv_na); check("se_memo", se_pct); check("se_no_aug", se_na)
check("z_rev", RT["z_memo"]); check("rel_effect_pct", rel_effect_pct)
check("total_ratio", total_ratio); check("total_ratio_1d", total_ratio)
check("per_branch_ratio", total_ratio * CTRL[C["branch"]].nunique() / CAMP[C["branch"]].nunique())

tbl = pd.DataFrame({"إيراد الخمسة": rev_t, "إيراد السبعة": rev_c, "النسبة (5÷7)": ratio})
tbl.index.name = "الشهر"
tbl["الفترة"] = ["حملة" if m in ACTIVE else ("قبل" if m in PRE else "بعد") for m in tbl.index]

noise = pd.DataFrame({
    "أساس كامل (10 أشهر)": [len(BASELINE), RT["mu_b"], cv_pct, se_pct, se_pct_cons],
    "دون الشهر الشاذ (أغسطس)": [len(BASE_NO_OUT), RT_NA["mu_b"], cv_na, se_na, RT_NA["sd_b"] * math.sqrt(1/2 + 1/len(BASE_NO_OUT)) / RT_NA["mu_b"] * 100],
}, index=pd.Index(["عدد الأشهر", "متوسط النسبة", "الانحراف المعياري %", "الخطأ المعياري لمتوسط شهرين % (اصطلاح المذكرة)",
                   "الخطأ المعياري % (التعريف المتحفظ)"], name="المقياس")).T

x = list(tbl.index)
emit("4_revenue_ratio", "مسار نسبة الإيراد والاتجاهات المتوازية والضجيج", [
    kpi([("التغير النسبي في النسبة", num(rel_effect_pct, 2, True) + "%", "warn"),
         ("الانحراف المعياري الشهري", num(cv_pct, 2) + "%", ""),
         ("الخطأ المعياري (شهران)", num(se_pct, 2) + "%", ""),
         ("المسافة بالخطأ المعياري", num(RT["z_memo"], 2), "bad" if abs(RT["z_memo"]) < 1 else "good")]),
    h2("نسبة الإيراد الشهرية"),
    svg_lines(x, {"نسبة الإيراد (5÷7)": ratio.values.tolist()}, ["#2a6f97"], "نسبة إيراد الخمسة إلى السبعة",
              band=(RT["mu_b"] - 2 * RT["sd_b"], RT["mu_b"] + 2 * RT["sd_b"]), shade=[x.index(m) for m in ACTIVE]),
    svg_lines(x, {"الخمسة (الحملة)": (rev_t / rev_t.iloc[0]).values.tolist(), "السبعة (الضابطة)": (rev_c / rev_c.iloc[0]).values.tolist()},
              ["#2a6f97", "#a9b4c2"], "الإيراد مؤشَّراً: تحرك المجموعتين معاً", shade=[x.index(m) for m in ACTIVE]),
    table_html(tbl, {"إيراد الخمسة": 0, "إيراد السبعة": 0, "النسبة (5÷7)": 4}),
    h2("تعريفا الضجيج جنباً إلى جنب"), table_html(noise, 3),
    note(f"قبل الحملة: نمت الخمسة {num(g_pre_t,2,True)}% والسبعة {num(g_pre_c,2,True)}% من {PRE[0]} إلى {PRE[-1]}، والنسبة بين "
         f"{num(pre.min(),2)} و{num(pre.max(),2)}: لا تباعد ظاهر بين المسارين، وهو شرط صلاحية فرق الفروق (تحقق قبل الحملة فقط). "
         f"أغسطس ({num(ratio.loc[OUTLIER_MONTH],2)}) أبعد شهر عن المتوسط ولا تفسير له في البيانات؛ أبقيناه في الحساب تحفظاً."),
], subtitle="الشريط الأحمر الفاتح = شهرا الحملة، والشريط الرمادي = متوسط الأساس ± انحرافين معياريين")

الشهر,إيراد الخمسة,إيراد السبعة,النسبة (5÷7),الفترة
2025-09,"1,710,936","1,061,496",1.6118,قبل
2025-10,"1,676,370","1,052,559",1.5927,قبل
2025-11,"1,795,094","1,099,764",1.6323,قبل
2025-12,"1,816,034","1,128,239",1.6096,قبل
2026-01,"1,518,803","951,857",1.5956,قبل
2026-02,"1,632,103","1,002,679",1.6277,قبل
2026-03,"2,502,979","1,529,182",1.6368,حملة
2026-04,"2,350,314","1,452,742",1.6178,حملة
2026-05,"1,820,675","1,104,289",1.6487,بعد
2026-06,"1,693,827","1,063,997",1.5919,بعد


saved: reports\html\analysis_report_4_revenue_ratio.html


## 5. متوسط قيمة الطلب (AOV)
نقيس الفرق الشهري بين المجموعتين (الخمسة − السبعة)، ثم بعد شهري الحملة عن الأساس بوحدة الخطأ المعياري. المتوسط الموزون وحده يُستخدم في المقارنات (لا متوسط المتوسطات).

In [6]:
# =====================================================================
# 5. متوسط قيمة الطلب: الفرق الشهري + المسافة عن الأساس
# =====================================================================
agg = (df.groupby([C["month"], C["in_camp"]])[[C["rev"], C["orders"]]].sum()
         .assign(aov=lambda t: t[C["rev"]] / t[C["orders"]]).reset_index())
pv = agg.pivot(index=C["month"], columns=C["in_camp"], values="aov")[[YES, NO]]
pv.columns = ["AOV الخمسة", "AOV السبعة"]
pv["الفرق (5 − 7)"] = pv["AOV الخمسة"] - pv["AOV السبعة"]
pv.index.name = "الشهر"

AT = distance_test(pv["الفرق (5 − 7)"], BASELINE, ACTIVE)
check("aov_base_mean", AT["mu_b"]); check("aov_base_sd", AT["sd_b"]); check("aov_act_mean", AT["mu_a"])
check("aov_dist", AT["dist"]); check("aov_se", AT["se_memo"]); check("aov_z", AT["z_memo"])

AOV4 = pd.DataFrame({"الأساس": [M_CAMP["aov_base"], M_CTRL["aov_base"]], "شهرا الحملة": [M_CAMP["aov_act"], M_CTRL["aov_act"]],
                     "النمو %": [M_CAMP["g_aov"] * 100, M_CTRL["g_aov"] * 100]},
                    index=pd.Index(["الخمسة (الحملة)", "السبعة (الضابطة)"], name="المجموعة"))

gap_series = pv["الفرق (5 − 7)"]
emit("5_aov", "متوسط قيمة الطلب (AOV)", [
    kpi([("فرق الفروق (نقطة مئوية)", num(DID.loc[A, "فرق الفروق (نقطة مئوية)"], 2, True), "warn"),
         ("المسافة عن الأساس (ريال)", num(AT["dist"], 2, True), ""),
         ("المسافة بالخطأ المعياري (المذكرة)", num(AT["z_memo"], 2), "bad" if abs(AT["z_memo"]) < 1 else "good"),
         ("المسافة بالخطأ المعياري (المتحفظ)", num(AT["z_cons"], 2), "bad" if abs(AT["z_cons"]) < 1 else "good")]),
    h2("الفرق الشهري بين المجموعتين"),
    svg_bars(list(gap_series.index), {"الفرق": gap_series.tolist()}, ["#a9b4c2"], "فرق متوسط قيمة الطلب بين المجموعتين",
             bar_colors=["#d1495b" if m in ACTIVE else "#a9b4c2" for m in gap_series.index], band=(AT["mu_b"] - 2 * AT["sd_b"], AT["mu_b"] + 2 * AT["sd_b"]), rot=1),
    table_html(pv, 2), h2("متوسط قيمة الطلب الموزون"), table_html(AOV4, 2),
    note(f"متوسط الفرق في الأساس {num(AT['mu_b'],2,True)} ريال (انحراف {num(AT['sd_b'],2)})، وفي شهري الحملة {num(AT['mu_a'],2,True)}. "
         f"فرق الشهر الواحد يتذبذب في الأشهر العادية بين {num(gap_series.loc[BASELINE].min(),2,True)} و{num(gap_series.loc[BASELINE].max(),2,True)} ريال، "
         "وشهرا الحملة متعاكسان (مارس مرتفع وأبريل منخفض). ضمن الضجيج: لا يمكن تمييز الفرق عن التذبذب الاعتيادي، ولا يعني ذلك غياب الأثر."),
], subtitle="الأحمر = شهرا الحملة، والشريط الرمادي = متوسط الأساس ± انحرافين معياريين")

الشهر,AOV الخمسة,AOV السبعة,الفرق (5 − 7)
2025-09,107.30,103.26,4.04
2025-10,114.05,116.02,-1.97
2025-11,106.60,111.26,-4.66
2025-12,111.17,110.51,0.65
2026-01,104.52,108.96,-4.44
2026-02,105.54,107.54,-2.00
2026-03,115.81,107.63,8.19
2026-04,106.73,109.89,-3.16
2026-05,105.64,98.92,6.72
2026-06,110.93,106.50,4.43


saved: reports\html\analysis_report_5_aov.html


## 6. عدد الطلبات
ندعم ادعاء «طلبات أكثر» أو ننقضه بالمقياس نفسه: نسبة طلبات الخمسة إلى طلبات السبعة، ومسافتها عن الأساس.

In [7]:
# =====================================================================
# 6. عدد الطلبات: النمو ونسبة الطلبات بين المجموعتين
# =====================================================================
mo = df.pivot_table(index=C["month"], columns=C["in_camp"], values=C["orders"], aggfunc="sum")
orders_ratio = (mo[YES] / mo[NO]).rename("نسبة الطلبات (5÷7)")
OT = distance_test(orders_ratio, BASELINE, ACTIVE)

check("ord_base_mean", OT["mu_b"]); check("ord_base_sd", OT["sd_b"]); check("ord_se", OT["se_memo"])
check("ord_act_mean", OT["mu_a"]); check("ord_dist", OT["dist"]); check("ord_z", OT["z_memo"])

otbl = pd.DataFrame({"طلبات الخمسة": mo[YES], "طلبات السبعة": mo[NO], "النسبة (5÷7)": orders_ratio})
otbl.index.name = "الشهر"
xo = list(otbl.index)
emit("6_orders", "عدد الطلبات", [
    kpi([("فرق الفروق: الطلبات (نقطة مئوية)", num(DID.loc[O, "فرق الفروق (نقطة مئوية)"], 2, True), "bad"),
         ("المسافة بالخطأ المعياري (المذكرة)", num(OT["z_memo"], 2), "warn"),
         ("المسافة بالخطأ المعياري (المتحفظ)", num(OT["z_cons"], 2), "warn")]),
    h2("نسبة الطلبات الشهرية"),
    svg_lines(xo, {"نسبة الطلبات (5÷7)": orders_ratio.values.tolist()}, ["#d1495b"], "نسبة طلبات الخمسة إلى طلبات السبعة",
              band=(OT["mu_b"] - 2 * OT["sd_b"], OT["mu_b"] + 2 * OT["sd_b"]), shade=[xo.index(m) for m in ACTIVE]),
    table_html(otbl, {"طلبات الخمسة": 0, "طلبات السبعة": 0, "النسبة (5÷7)": 4}),
    note(f"نمت طلبات الخمسة {num(M_CAMP['g_orders']*100,2)}% والسبعة {num(M_CTRL['g_orders']*100,2)}%. التقدير النقطي للفرق سالب لكنه ضمن الضجيج: "
         "لا دليل على زيادة طلبات تتجاوز ما شهدته الشبكة كلها، ولا دليل على نقصها كذلك."),
])

الشهر,طلبات الخمسة,طلبات السبعة,النسبة (5÷7)
2025-09,"15,945","10,280",1.5511
2025-10,"14,698","9,072",1.6201
2025-11,"16,840","9,885",1.7036
2025-12,"16,336","10,209",1.6002
2026-01,"14,531","8,736",1.6633
2026-02,"15,464","9,324",1.6585
2026-03,"21,612","14,208",1.5211
2026-04,"22,022","13,220",1.6658
2026-05,"17,234","11,163",1.5439
2026-06,"15,270","9,991",1.5284


saved: reports\html\analysis_report_6_orders.html


## 7. عتبات الكشف ومدة التجربة ونقطة التعادل
- **أصغر أثر نسبي يمكن تمييزه** بشهرين = k × SE، حيث k ≈ 2 (ثقة 95%، قدرة ≈ 50%) أو ≈ 2.8 (ثقة 95% وقدرة 80%).
- **الأشهر اللازمة** لكشف أثر e: n = 2 × (k × SE₂ ÷ e)² حيث SE₂ الخطأ المعياري لمتوسط شهرين، بافتراض استقلال الأشهر (الأشهر المتجاورة تتشابه فعلاً، فالمدة الفعلية أطول).
- **نقطة التعادل (%)** = (تكلفة الحملة ÷ هامش الربح) ÷ إيراد الفروع المشمولة خلال مدة الحملة.

In [8]:
# =====================================================================
# 7. عتبات الكشف + جدول مدة التجربة (6-1) + دالة نقطة التعادل
# =====================================================================
SE2 = se_pct                                              # %
thr_conf, thr_p80 = K_CONF * SE2, K_POWER80 * SE2
thr_na = K_CONF * se_na
check("thr_conf", thr_conf); check("thr_power80", thr_p80); check("thr_no_aug", thr_na)

def months_needed(effect_pct, k, se2=SE2):
    return math.ceil(2 * (k * se2 / effect_pct) ** 2 - 1e-9)

EFFECTS = [3.0, 2.0, 1.5]
dur = pd.DataFrame({"حد أدنى (ثقة 95%، قدرة ≈50%)": [months_needed(e, K_CONF) for e in EFFECTS],
                    "معيار مقبول (ثقة 95%، قدرة 80%)": [months_needed(e, K_POWER80) for e in EFFECTS]},
                   index=pd.Index([f"{e:g}%" for e in EFFECTS], name="الأثر النسبي المراد كشفه"))
for key, e in (("3", 3.0), ("2", 2.0), ("15", 1.5)):
    check("n_conf_" + key, months_needed(e, K_CONF)); check("n_pow_" + key, months_needed(e, K_POWER80))

def breakeven_pct(cost, margin, covered_revenue_in_campaign):
    """نقطة التعادل (% من إيراد الفروع المشمولة): تُدخَل التكلفة والهامش من التسويق والمالية."""
    return cost / margin / covered_revenue_in_campaign * 100

COST, MARGIN = None, None          # ← يُملآن من إدارتي التسويق والمالية (ريال، نسبة عشرية مثل 0.30)
cov_rev = M_CAMP["rev_base"] * len(ACTIVE)                # إيراد الفروع المشمولة المتوقع خلال مدة الحملة (على الأساس)
be_card = (kpi([("نقطة التعادل", num(breakeven_pct(COST, MARGIN, cov_rev), 2) + "%", "")])
           if COST and MARGIN else note("نقطة التعادل غير محسوبة: أدخلي COST و MARGIN في هذه الخلية (تُطلبان من إدارتي التسويق والمالية)."))

emit("7_detection", "عتبات الكشف ومدة التجربة", [
    kpi([("الخطأ المعياري لمتوسط شهرين", num(SE2, 2) + "%", ""),
         ("أصغر أثر ممكن التمييز بشهرين", num(thr_conf, 1) + "%", "warn"),
         ("بقدرة 80%", num(thr_p80, 1) + "%", "bad"),
         ("دون أغسطس", num(thr_na, 1) + "%", "")]),
    h2("الجدول 6-1: الأشهر اللازمة لكشف أثر نسبي في الإيراد"), table_html(dur, 0),
    note("الافتراضات: الخطأ المعياري لمتوسط شهرين مقدَّر من عشرة أشهر منها أغسطس الشاذ؛ استقلال الأشهر؛ التصميم الحالي (5 فروع مقابل 7). "
         "الأرقام تقريبية؛ المدة الفعلية أطول لتشابه الأشهر المتجاورة وأقصر كلما زاد عدد الفروع. تتناسب المدة مع مقلوب مربع الأثر: نصف الأثر يتطلب نحو أربعة أضعاف المدة."),
    h2("نقطة التعادل"), be_card,
])

الأثر النسبي المراد كشفه,حد أدنى (ثقة 95%، قدرة ≈50%),معيار مقبول (ثقة 95%، قدرة 80%)
3%,3,5
2%,5,10
1.5%,9,17


saved: reports\html\analysis_report_7_detection.html


## 8. نمو الإيراد لكل فرع (١٢ فرعاً)
نمو كل فرع = متوسط شهري الحملة ÷ متوسط الأشهر العشرة − 1. متوسط الفرع (غير الموزون) قد يختلف عن نمو المجموعة (الموزون بالحجم)، وهذا متوقع وليس خطأ.

In [9]:
# =====================================================================
# 8. نمو الإيراد لكل فرع + مقارنة نطاقي المجموعتين
# =====================================================================
per = np.where(df[C["month"]].isin(ACTIVE), "act", "base")
g = df.assign(period=per).groupby([C["branch"], C["in_camp"], "period"])[C["rev"]].mean().unstack("period")
bg = ((g["act"] / g["base"] - 1) * 100).rename("نمو الإيراد %").reset_index()
bg.columns = ["الفرع", "ضمن الحملة", "نمو الإيراد %"]
bg = bg.sort_values("نمو الإيراد %").reset_index(drop=True)
bg.insert(0, "الترتيب", bg.index + 1)

gt_ = bg.loc[bg["ضمن الحملة"] == YES, "نمو الإيراد %"]
gc_ = bg.loc[bg["ضمن الحملة"] == NO, "نمو الإيراد %"]
check("camp_min", gt_.min()); check("camp_max", gt_.max()); check("ctrl_min", gc_.min()); check("ctrl_max", gc_.max())

colors = ["#2a6f97" if v == YES else "#a9b4c2" for v in bg["ضمن الحملة"]]
emit("8_branches", "نمو الإيراد لكل فرع", [
    kpi([("متوسط نمو فروع الحملة", num(gt_.mean(), 2) + "%", ""), ("متوسط نمو الضابطة", num(gc_.mean(), 2) + "%", ""),
         ("نطاق الخمسة", f"{num(gt_.min(),1)} – {num(gt_.max(),1)}", ""), ("نطاق السبعة", f"{num(gc_.min(),1)} – {num(gc_.max(),1)}", "")]),
    svg_hbars(bg["الفرع"].tolist(), bg["نمو الإيراد %"].tolist(), colors, "نمو الإيراد لكل فرع"),
    table_html(bg.set_index("الفرع"), {"الترتيب": 0, "نمو الإيراد %": 2}),
    note(f"تداخل الترتيب كامل: أعلى فرعين نمواً هما {' و'.join(bg.tail(2)['الفرع'])} (أحدهما من الضابطة). "
         "التشتت داخل كل مجموعة أكبر بكثير من الفرق بين المتوسطين، فلا يظهر فرع منفرد يغيّر الاستنتاج."),
])

الفرع,الترتيب,ضمن الحملة,نمو الإيراد %
الخبر ١,1,لا,37.68
جدة ١,2,نعم,38.20
الرياض ٢,3,نعم,38.79
مكة ١,4,لا,39.60
المدينة ١,5,لا,40.61
جدة ٢,6,نعم,40.69
تبوك ١,7,لا,42.29
بريدة ١,8,لا,43.77
أبها ١,9,لا,45.68
الرياض ١,10,نعم,46.80


saved: reports\html\analysis_report_8_branches.html


## 9. اختبارات تكميلية (اختيارية)
**معطّلة افتراضياً** كي يبقى الدفتر مطابقاً لنص المذكرة ("لم تُجرَ اختبارات دلالة رسمية"). فعّلي `RUN_SENSITIVITY = True` في أول خلية لعرضها. ومع n = 2 شهراً فعّالاً تبقى قدرتها منخفضة جداً؛ فعدم الدلالة ليس دليلاً على غياب الأثر.

In [10]:
# =====================================================================
# 9. اختبارات حساسية تكميلية (Welch / Permutation / Mann–Whitney) — اختيارية
# =====================================================================
def permutation_p(series, n_act, observed):
    """اختبار تبديل دقيق: كل تجميعات n_act شهراً من الأشهر الاثني عشر."""
    v = series.values
    diffs = [v[list(ix)].mean() - np.delete(v, list(ix)).mean() for ix in itertools.combinations(range(len(v)), n_act)]
    return float(np.mean(np.abs(diffs) >= abs(observed) - 1e-12)), len(diffs)

if RUN_SENSITIVITY:
    from scipy import stats
    rows = []
    for name, ser, T in (("نسبة الإيراد (5÷7)", ratio, RT), ("فرق AOV (5 − 7)", pv["الفرق (5 − 7)"], AT), ("نسبة الطلبات (5÷7)", orders_ratio, OT)):
        p, n_comb = permutation_p(ser, len(ACTIVE), T["dist"])
        w = stats.ttest_ind(ser.loc[ACTIVE], ser.loc[BASELINE], equal_var=False)
        rows.append([name, T["z_memo"], T["z_cons"], w.pvalue, p])
    mw = stats.mannwhitneyu(gt_, gc_, alternative="two-sided")
    slope = stats.linregress(np.arange(len(PRE)), ratio.loc[PRE].values)
    sens = pd.DataFrame(rows, columns=["المقياس", "المسافة/SE (المذكرة)", "المسافة/SE (المتحفظ)", "Welch p", "Permutation p"]).set_index("المقياس")
    emit("9_sensitivity", "اختبارات حساسية تكميلية", [
        table_html(sens, 3),
        kpi([("Mann–Whitney p (نمو الفروع)", num(mw.pvalue, 3), ""), ("ميل النسبة قبل الحملة (لكل شهر)", num(slope.slope, 5, True), ""),
             ("p لميل ما قبل الحملة", num(slope.pvalue, 3), "")]),
        note(f"اختبار التبديل دقيق على {n_comb} تجميعة. هذه فحوص تكميلية للاطلاع؛ الاستنتاجات الرئيسية مبنية على مقارنة التغير بالخطأ المعياري."),
    ])
else:
    print("sensitivity tests skipped (RUN_SENSITIVITY = False)")

sensitivity tests skipped (RUN_SENSITIVITY = False)


## 10. تدقيق التطابق مع المذكرة + الملخص التنفيذي
يقارن كل رقم محسوب بقيمته في «سجل الأرقام» (القسم 0) بعد تقريبه إلى منازل المذكرة. يتوقف التشغيل إن اختلف رقم واحد.

In [11]:
# =====================================================================
# 10. تدقيق التطابق (Consistency Audit) + الملخص التنفيذي
# =====================================================================
audit = pd.DataFrame(CHECKS)
n_ok, n_all = int(audit["ok"].sum()), len(audit)
rows_html = "".join(
    f'<tr><td class="lbl">{html.escape(r.label)}</td><td>{html.escape(r.section)}</td>'
    f'<td class="num">{r.memo:,}</td><td class="num">{r.got:,}</td>'
    f'<td class="{"ok" if r.ok else "no"}">{"✔" if r.ok else "✘"}</td></tr>' for r in audit.itertuples())
audit_table = ('<div class="scroll"><table><thead><tr><th>الرقم</th><th>القسم في المذكرة</th><th>المذكرة</th><th>المحسوب</th><th>الحالة</th></tr></thead>'
               f'<tbody>{rows_html}</tbody></table></div>')

emit("10_audit", "تدقيق التطابق مع المذكرة", [
    kpi([("أرقام مطابقة", f'{num(n_ok,0)} / {num(n_all,0)}', "good" if n_ok == n_all else "bad")]), audit_table])

emit("0_executive_summary", "الملخص التنفيذي: حملة «ربيع القهوة»", [
    kpi([("الخمسة بطريقة التقرير", num(M12_CAMP["g_rev"] * 100, 2) + "%", "warn"),
         ("السبعة بطريقة التقرير (بلا حملة)", num(M12_CTRL["g_rev"] * 100, 2) + "%", "warn"),
         ("فرق الفروق بالأساس المصحَّح", num(DID.loc[R, "فرق الفروق (نقطة مئوية)"], 2, True) + " نقطة", "warn"),
         ("المسافة بالخطأ المعياري", num(RT["z_memo"], 2), "bad"),
         ("أصغر أثر ممكن التمييز بشهرين", num(thr_conf, 1) + "%", "")]),
    h2("التفكيك"), table_html(DID, 3),
    note("<b>القراءة الصريحة:</b> بيانات التقرير لا تدعم نسب ارتفاع الـ33% إلى الحملة. بعد تصحيح الأساس ومقارنة الخمسة بالسبعة التي لم تتلقَّ الحملة، "
         "الفرق نحو نصف نقطة مئوية، وهو أصغر من تذبذب المقياس. هذا لا يعني أن الحملة بلا أثر، فبشهرين لا يُكشف إلا أثر يتجاوز نحو 3%. "
         "والخفض العام للأسعار يمنع الحكم على الحملة منفردة."),
    "<p>التقارير التفصيلية: <code>reports/html/analysis_report_*.html</code></p>",
])

assert n_ok == n_all, "numbers differ from the memo:\n" + audit.loc[~audit["ok"], ["label", "memo", "got"]].to_string()
print(f"audit passed: {n_ok}/{n_all} numbers match the memo")

الرقم,القسم في المذكرة,المذكرة,المحسوب,الحالة
عدد الصفوف,٨,144.0,144.0,✔
أقصى فرق بين الإيراد وحاصل (الطلبات × متوسط الطلب) بالريال,٨,20.4,20.4,✔
عدد الفروع المشمولة بخفض السعر العام,٨,12.0,12.0,✔
أساس الأشهر العشرة للخمسة (ريال),٤,"1,699,846.0","1,699,846.0",✔
متوسط شهري الحملة للخمسة (ريال),٤,"2,426,647.0","2,426,647.0",✔
أساس الأشهر العشرة للسبعة (ريال),٤,"1,048,445.0","1,048,445.0",✔
متوسط شهري الحملة للسبعة (ريال),٤,"1,490,962.0","1,490,962.0",✔
نمو إيراد الخمسة بالأساس المصحَّح (%),٤,42.76,42.76,✔
نمو إيراد السبعة بالأساس المصحَّح (%),٤,42.21,42.21,✔
فرق الفروق للإيراد (نقطة مئوية),١,0.55,0.55,✔


saved: reports\html\analysis_report_10_audit.html


المقياس,نمو الحملة %,نمو الضابطة %,فرق الفروق (نقطة مئوية),الأثر النسبي %,الأثر اللوغاريتمي ×100
الإيراد,42.757,42.207,0.550,0.387,0.386
عدد الطلبات,38.510,41.153,-2.643,-1.872,-1.890
متوسط قيمة الطلب,3.066,0.747,2.319,2.302,2.276


saved: reports\html\analysis_report_0_executive_summary.html
audit passed: 85/85 numbers match the memo
